# 11 — cdfmm backends versus FMM3D

This notebook compares cdfmm CUDA-full, CUDA-partial, and oneMKL CPU-static with FMM3D 2.1.0 for source-point dipole fields. Every particle is both a source and its corresponding target, and all implementations exclude the singular self interaction. FMM3D returns $\nabla\phi$, so its result is converted with $H=-\nabla\phi$.

The main sweep covers particle counts 10,000–30,000, Cartesian orders 4, 6, and 8, tree depths 2–4, and FMM3D tolerances $10^{-3}$ and $10^{-4}$. cdfmm plans are constructed and warmed before timing; plan construction, matrix creation, immutable geometry upload, and the first CUDA-full identity upload are excluded from runtime measurements. Plans are released one at a time, so CUDA-full and CUDA-partial are never simultaneously resident on the device.

Build the required CUDA/oneMKL Python extension and install the pinned comparison dependency from the repository root before running:

```console
conda env update -n cdfmm -f environment.yml
conda env update -n cdfmm -f environment-cuda.yml
conda env update -n cdfmm -f environment-fmm3d.yml
conda activate cdfmm
cmake --fresh --preset notebooks
cmake --build --preset notebooks -j
ctest --preset notebooks
./examples/notebooks/install_fmm3d.sh
```

In [ ]:
# Particle counts used to test how runtime and accuracy scale with N.
PARTICLE_COUNTS = [10_000, 20_000, 30_000]
# Total-degree Cartesian expansion orders used by cdfmm.
EXPANSION_ORDERS = [4, 6, 8]
# Maximum levels of the complete uniform cdfmm tree.
TREE_DEPTHS = [2, 3, 4]
# Requested FMM3D relative tolerances; FMM3D selects its own expansion order.
FMM3D_EPS_VALUES = [1.0e-3, 1.0e-4]
# Number of warmed, synchronous evaluations used for each runtime median.
TIMING_REPEATS = 3
# Untimed evaluations used to initialise CUDA, oneMKL, and lazy plan state.
WARMUP_EVALUATIONS = 1
# Deterministic targets checked against exact direct P2P for each geometry.
ACCURACY_TARGETS = 128
# Changing-moment counts for the separate representative throughput test.
REPEAT_COUNTS = [10, 50, 100]
# FMM3D natively supports multiple densities through its `nd` argument.
# VECTOR_BATCH_SIZE is added by this notebook: it limits how many moment
# states are passed to one native multi-density call to bound host memory.
# It is not an FMM3D accuracy or tree-construction parameter.
VECTOR_BATCH_SIZE = 25
# Representative grid point used only for changing-moment throughput.
THROUGHPUT_PARTICLES = 20_000
THROUGHPUT_ORDER = 6
THROUGHPUT_DEPTH = 3
THROUGHPUT_FMM3D_EPS = 1.0e-3
# Reproducible geometry/moment seed and shared CPU thread count.
RANDOM_SEED = 314159
CPU_THREADS = 8

import os
# OpenMP reads this before FMM3D and oneMKL are imported.
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)

In [ ]:
import gc
import importlib.metadata
import platform
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

import cdfmm
try:
    import fmm3dpy
except ImportError as error:
    raise RuntimeError(
        "FMM3D is missing. Run ./examples/notebooks/install_fmm3d.sh "
        "from the repository root, then restart this kernel."
    ) from error

try:
    from fmm3d_comparison import (
        fmm3d_laplace_nterms, fmm3d_source_fields,
        relative_error_metrics, timed_call, validate_source_point_geometry,
    )
except ModuleNotFoundError:
    from examples.notebooks.fmm3d_comparison import (
        fmm3d_laplace_nterms, fmm3d_source_fields,
        relative_error_metrics, timed_call, validate_source_point_geometry,
    )

missing = []
if not cdfmm.cuda_full_available():
    missing.append("CUDA_FULL (rebuild with CUDA full support)")
if not cdfmm.cuda_m2l_p2p_available():
    missing.append("CUDA_PARTIAL (rebuild with CUDA support)")
if not cdfmm.one_mkl_available():
    missing.append("oneMKL CPU-static (reconfigure with oneMKL)")
if missing:
    raise RuntimeError(
        "Required comparison backends are unavailable: " + "; ".join(missing) + ". "
        "Run the notebook build commands above, then restart this kernel."
    )

FMM3D_SOURCE_VERSION = "2.1.0"
fmm3d_distribution_version = importlib.metadata.version("fmm3dpy")
configuration = {
    "particles": PARTICLE_COUNTS,
    "cdfmm orders/depths": f"{EXPANSION_ORDERS}/{TREE_DEPTHS}",
    "FMM3D eps/nterms": {
        eps: fmm3d_laplace_nterms(eps) for eps in FMM3D_EPS_VALUES
    },
    "timed evaluations per case": TIMING_REPEATS,
    "CPU threads": CPU_THREADS,
    "CUDA device": cdfmm.cuda_device_description(),
    "FMM3D source/distribution": (
        f"{FMM3D_SOURCE_VERSION}/{fmm3d_distribution_version}"
    ),
    "host": platform.node(),
}
configuration

## Measurement contract

For cdfmm, every plan is constructed outside the timer. This excludes tree creation, static matrix generation, persistent buffer allocation, and immutable CPU-to-GPU transfers. At least one untimed evaluation is then performed; this also excludes CUDA-full's first-use identity upload and runtime initialisation. Only subsequent synchronous `plan.evaluate(...)` calls are timed. Dynamic moment transfers and result transfers remain included because they are required for every evaluation.

FMM3D's Python API exposes `lfmm3d` as a complete one-shot call and does not expose a reusable geometry plan. Its timed call therefore necessarily includes whatever internal setup FMM3D performs. The comparison reports this interface-level distinction rather than attempting to subtract an unobservable component.

In [ ]:
BACKENDS = [
    ("cdfmm CUDA-full", cdfmm.ExecutionBackend.CUDA_FULL, None),
    ("cdfmm CUDA-partial", cdfmm.ExecutionBackend.CUDA_PARTIAL, None),
    ("cdfmm oneMKL CPU-static", cdfmm.ExecutionBackend.CPU_STATIC,
     cdfmm.StaticMatrixBackend.ONE_MKL),
]

def make_problem(particle_count):
    rng = np.random.default_rng(RANDOM_SEED + particle_count)
    positions = rng.uniform(-0.95, 0.95, size=(particle_count, 3))
    moments = rng.normal(size=(particle_count, 3))
    moments /= np.linalg.norm(moments, axis=1, keepdims=True)
    identities = np.arange(particle_count, dtype=np.int64)
    validate_source_point_geometry(positions, positions, identities)
    return positions, moments, identities, np.asfortranarray(positions.T)

def make_cdfmm_plan(positions, order, depth, backend, matrix_backend=None):
    options = cdfmm.UniformFmmOptions()
    options.expansion_order = order
    options.tree.max_level = depth
    options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    options.tree.root_half_width = 1.0
    options.backend = backend
    if matrix_backend is not None:
        options.static_matrix_backend = matrix_backend
    return cdfmm.UniformFmm(positions, positions, options)

def evaluate_cdfmm(plan, moments, identities):
    return plan.evaluate(
        moments, output="field", target_source_indices=identities,
    )["H"]

def evaluate_fmm3d(fmm3d_sources, moments, eps):
    output = fmm3dpy.lfmm3d(
        eps=eps, sources=fmm3d_sources,
        dipvec=np.asfortranarray(moments.T), pg=2,
    )
    fields = fmm3d_source_fields(output)
    if fields.shape != moments.shape:
        raise RuntimeError(f"FMM3D source evaluation returned {fields.shape}")
    return fields

def exact_sample_fields(positions, moments, sample_indices):
    fields = np.empty((len(sample_indices), 3))
    for row, particle in enumerate(sample_indices):
        fields[row] = cdfmm.p2p_dipole_sum(
            positions[particle], positions, moments, self_index=int(particle),
        )["H"]
    return fields

def median_timed_evaluations(function):
    result = None
    samples = []
    for _ in range(TIMING_REPEATS):
        result, seconds = timed_call(function)
        samples.append(seconds)
    return result, float(np.median(samples)), samples

def release_plan(plan):
    return None

## Accuracy and reused-evaluation runtime sweep

The cdfmm sweep evaluates the full particle/order/depth Cartesian product for each backend. FMM3D is evaluated once per particle-count/tolerance pair because cdfmm order and tree depth do not apply to FMM3D. Exact direct fields are computed for the same deterministic sample once per particle count.

In [ ]:
#WARNING: Takes roughly 50min to run on a workstation with 8 CPU threads and a single NVIDIA 5090 GPU.

comparison_rows = []
for particle_count in PARTICLE_COUNTS:
    print(f"Preparing N={particle_count:,} geometry...", flush=True)
    positions, base_moments, identities, fmm3d_sources = make_problem(
        particle_count
    )
    sample_indices = np.linspace(
        0, particle_count - 1, min(ACCURACY_TARGETS, particle_count), dtype=int
    )
    exact_fields = exact_sample_fields(
        positions, base_moments, sample_indices
    )

    for eps in FMM3D_EPS_VALUES:
        print(f"  FMM3D eps={eps:g}", flush=True)
        for _ in range(WARMUP_EVALUATIONS):
            _ = evaluate_fmm3d(fmm3d_sources, base_moments, eps)
        fields, median_seconds, samples = median_timed_evaluations(
            lambda: evaluate_fmm3d(fmm3d_sources, base_moments, eps)
        )
        metrics = relative_error_metrics(
            fields[sample_indices], exact_fields
        )
        comparison_rows.append({
            "particles": particle_count, "implementation": "FMM3D",
            "order": None, "depth": None, "eps": eps,
            "median_seconds": median_seconds, "samples": samples,
            "rms_error": metrics.rms, "max_error": metrics.maximum,
        })

    for order in EXPANSION_ORDERS:
        for depth in TREE_DEPTHS:
            for label, backend, matrix_backend in BACKENDS:
                print(
                    f"  {label}: order={order}, depth={depth}", flush=True
                )
                # Construction performs matrix creation and immutable GPU
                # uploads. It is deliberately outside all runtime samples.
                plan = make_cdfmm_plan(
                    positions, order, depth, backend, matrix_backend
                )
                # The warm-up also performs CUDA-full's lazy identity upload.
                for _ in range(WARMUP_EVALUATIONS):
                    _ = evaluate_cdfmm(plan, base_moments, identities)
                fields, median_seconds, samples = median_timed_evaluations(
                    lambda: evaluate_cdfmm(plan, base_moments, identities)
                )
                metrics = relative_error_metrics(
                    fields[sample_indices], exact_fields
                )
                comparison_rows.append({
                    "particles": particle_count, "implementation": label,
                    "order": order, "depth": depth, "eps": None,
                    "median_seconds": median_seconds, "samples": samples,
                    "rms_error": metrics.rms,
                    "max_error": metrics.maximum,
                })
                plan = release_plan(plan)
                gc.collect()

    del positions, base_moments, identities, fmm3d_sources, exact_fields
    gc.collect()

In [ ]:
print(
    f"{'N':>7s} {'Implementation':27s} {'p':>3s} {'d':>3s} "
    f"{'eps':>9s} {'median (s)':>12s} {'RMS error':>12s} {'Max error':>12s}"
)
for row in comparison_rows:
    order = "-" if row["order"] is None else str(row["order"])
    depth = "-" if row["depth"] is None else str(row["depth"])
    eps = "-" if row["eps"] is None else f"{row['eps']:.0e}"
    print(
        f"{row['particles']:7d} {row['implementation']:27s} "
        f"{order:>3s} {depth:>3s} {eps:>9s} "
        f"{row['median_seconds']:12.6f} {row['rms_error']:12.4e} "
        f"{row['max_error']:12.4e}"
    )

In [ ]:
for particle_count in PARTICLE_COUNTS:
    fig, axes = plt.subplots(2, len(EXPANSION_ORDERS), figsize=(17, 8))
    for column, order in enumerate(EXPANSION_ORDERS):
        runtime_axis = axes[0, column]
        accuracy_axis = axes[1, column]
        for label, _, _ in BACKENDS:
            selected = sorted(
                (row for row in comparison_rows
                 if row["particles"] == particle_count
                 and row["implementation"] == label
                 and row["order"] == order),
                key=lambda row: row["depth"],
            )
            depths = [row["depth"] for row in selected]
            runtime_axis.plot(
                depths, [row["median_seconds"] for row in selected],
                marker="o", label=label,
            )
            accuracy_axis.plot(
                depths, [row["rms_error"] for row in selected],
                marker="o", label=label,
            )
        fmm_rows = [
            row for row in comparison_rows
            if row["particles"] == particle_count
            and row["implementation"] == "FMM3D"
        ]
        for row in fmm_rows:
            fmm_label = f"FMM3D eps={row['eps']:.0e}"
            runtime_axis.axhline(
                row["median_seconds"], linestyle="--", label=fmm_label
            )
            accuracy_axis.axhline(
                row["rms_error"], linestyle="--", label=fmm_label
            )
        runtime_axis.set_title(f"order={order}")
        runtime_axis.set_ylabel("Reused evaluation (s)")
        accuracy_axis.set_ylabel("RMS relative field error")
        for axis in (runtime_axis, accuracy_axis):
            axis.set_xlabel("Tree depth")
            axis.set_xticks(TREE_DEPTHS)
            axis.set_yscale("log")
            axis.grid(True, which="both", alpha=0.25)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"cdfmm reused evaluations versus FMM3D, N={particle_count:,}")
    fig.tight_layout()
    plt.show()

## Changing-moment throughput at one representative grid point

Moment generation and cdfmm plan construction remain outside the timers. Each cdfmm method reuses one persistent plan. Ordinary FMM3D calls represent sequential updates. The vectorized FMM3D curve uses FMM3D's native multi-density `nd` interface, while `VECTOR_BATCH_SIZE` is only this notebook's memory-bounding chunk size. Vectorized batches are therefore a throughput mode, not a reusable sequential-plan API.

In [ ]:
throughput_positions, throughput_base_moments, throughput_identities, throughput_fmm3d_sources = make_problem(
    THROUGHPUT_PARTICLES
)

def make_moment_states(count):
    state_rng = np.random.default_rng(RANDOM_SEED + 1 + count)
    return state_rng.normal(size=(count, THROUGHPUT_PARTICLES, 3))

def time_cdfmm_states(states, backend, matrix_backend):
    plan = make_cdfmm_plan(
        throughput_positions, THROUGHPUT_ORDER, THROUGHPUT_DEPTH,
        backend, matrix_backend,
    )
    _ = evaluate_cdfmm(plan, states[0], throughput_identities)
    checksum = 0.0
    start = perf_counter()
    for moments in states:
        checksum += float(
            evaluate_cdfmm(plan, moments, throughput_identities)[0, 0]
        )
    seconds = perf_counter() - start
    plan = release_plan(plan)
    gc.collect()
    return seconds, checksum

def time_fmm3d_calls(states):
    checksum = 0.0
    start = perf_counter()
    for moments in states:
        checksum += float(evaluate_fmm3d(
            throughput_fmm3d_sources, moments, THROUGHPUT_FMM3D_EPS
        )[0, 0])
    return perf_counter() - start, checksum

def time_fmm3d_vectorized(states):
    checksum = 0.0
    start = perf_counter()
    for begin in range(0, len(states), VECTOR_BATCH_SIZE):
        batch = states[begin:begin + VECTOR_BATCH_SIZE]
        output = fmm3dpy.lfmm3d(
            eps=THROUGHPUT_FMM3D_EPS, sources=throughput_fmm3d_sources,
            dipvec=np.asfortranarray(np.transpose(batch, (0, 2, 1))),
            pg=2, nd=len(batch),
        )
        fields = fmm3d_source_fields(output)
        checksum += float(np.sum(fields[:, 0, 0]))
    return perf_counter() - start, checksum

performance_rows = []
for count in REPEAT_COUNTS:
    print(f"Benchmarking {count} changing moment states...", flush=True)
    states = make_moment_states(count)
    for label, backend, matrix_backend in BACKENDS:
        seconds, checksum = time_cdfmm_states(states, backend, matrix_backend)
        performance_rows.append((label, count, seconds, checksum))
    for label, timer in [
        ("FMM3D repeated calls", time_fmm3d_calls),
        ("FMM3D vectorized batches", time_fmm3d_vectorized),
    ]:
        seconds, checksum = timer(states)
        performance_rows.append((label, count, seconds, checksum))
    del states
    gc.collect()

for method, count, seconds, checksum in performance_rows:
    if not np.isfinite(checksum):
        raise RuntimeError(f"{method} produced a non-finite checksum")
print(f"{'Method':30s} {'States':>8s} {'Total (s)':>12s} {'s/state':>12s} {'states/s':>12s}")
for method, count, seconds, _ in performance_rows:
    print(f"{method:30s} {count:8d} {seconds:12.6f} {seconds/count:12.6e} {count/seconds:12.3f}")

In [ ]:
methods = list(dict.fromkeys(row[0] for row in performance_rows))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for method in methods:
    selected = [row for row in performance_rows if row[0] == method]
    counts = np.array([row[1] for row in selected])
    totals = np.array([row[2] for row in selected])
    axes[0].plot(counts, totals, marker="o", label=method)
    axes[1].plot(counts, totals / counts, marker="o", label=method)
    axes[2].plot(counts, counts / totals, marker="o", label=method)
for axis in axes:
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel("Changing moment states")
    axis.grid(True, which="both", alpha=0.25)
axes[0].set_ylabel("Total wall time (s)")
axes[1].set_ylabel("Mean seconds per state")
axes[2].set_ylabel("States per second")
axes[0].legend(fontsize=8)
fig.suptitle(
    f"Changing moments at N={THROUGHPUT_PARTICLES:,}, "
    f"order={THROUGHPUT_ORDER}, depth={THROUGHPUT_DEPTH}"
)
fig.tight_layout()
plt.show()

The main runtime curves represent steady-state cdfmm evaluations after all reusable setup and static transfers. FMM3D timings represent its public one-shot call because no reusable geometry-plan interface is available. Compare accuracy alongside runtime: cdfmm order/depth and FMM3D `eps` are independent controls and are not assumed to be accuracy-equivalent. The vectorized FMM3D curve measures multi-density throughput and should not be interpreted as the cost of one sequential moment update.